In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv('data10.csv')

In [3]:
df.head()

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,category
0,119646801,17,16,597,768,293,0,35.120,97.06,384,...,20,45580.5,32287.848,131880,32165,9922286.0,281712.56,10014561,9030908,0
1,375,2,1,38,0,38,0,19.000,26.88,0,...,20,0.0,0.000,0,0,0.0,0.00,0,0,0
2,281,3,0,46,0,46,0,15.336,26.56,0,...,20,0.0,0.000,0,0,0.0,0.00,0,0,0
3,99999555,2,2,16,0,8,8,8.000,0.00,0,...,32,1.0,0.000,1,1,99999550.0,0.00,99999553,99999553,4
4,10070086,4,4,379,408,379,0,94.750,189.50,408,...,20,0.0,0.000,0,0,0.0,0.00,0,0,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033931 entries, 0 to 1033930
Data columns (total 69 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Flow Duration      1033931 non-null  int64  
 1   Tot Fwd Pkts       1033931 non-null  int64  
 2   Tot Bwd Pkts       1033931 non-null  int64  
 3   TotLen Fwd Pkts    1033931 non-null  int64  
 4   TotLen Bwd Pkts    1033931 non-null  int64  
 5   Fwd Pkt Len Max    1033931 non-null  int64  
 6   Fwd Pkt Len Min    1033931 non-null  int64  
 7   Fwd Pkt Len Mean   1033931 non-null  float64
 8   Fwd Pkt Len Std    1033931 non-null  float64
 9   Bwd Pkt Len Max    1033931 non-null  int64  
 10  Bwd Pkt Len Min    1033931 non-null  int64  
 11  Bwd Pkt Len Mean   1033931 non-null  float64
 12  Bwd Pkt Len Std    1033931 non-null  float64
 13  Flow Byts/s        1033931 non-null  float64
 14  Flow Pkts/s        1033931 non-null  float64
 15  Flow IAT Mean      1033931 non-n

In [5]:
df['category'].value_counts() # 0 represents benign records and 1, 2, 3, 4, 5, 6 represent attack records of different types

0    901347
3     77595
4     19657
1     14453
5     11382
2      9410
6        87
Name: category, dtype: int64

In [6]:
X, y = df.iloc[:, :-1], df.iloc[:, -1]

In [7]:
from collections import Counter
Counter(y) # finding count of all labels

Counter({0: 901347, 4: 19657, 3: 77595, 1: 14453, 5: 11382, 2: 9410, 6: 87})

In [8]:
# CASE 1
# finding required counts of all labels
over_sampling = round((y.value_counts()[0]/y.value_counts()[1:].sum())*y.value_counts()[1:]).to_dict()
over_sampling = {key: int(value) for key, value in over_sampling.items()}
over_sampling

C:\Users\DELL\AppData\Local\Temp\ipykernel_14012\2526601099.py:3: FutureWarning: The behavior of `series[i:j]` with an integer-dtype index is deprecated. In a future version, this will be treated as *label-based* indexing, consistent with e.g. `series[i]` lookups. To retain the old behavior, use `series.iloc[i:j]`. To get the future behavior, use `series.loc[i:j]`.
  over_sampling = round((y.value_counts()[0]/y.value_counts()[1:].sum())*y.value_counts()[1:]).to_dict()
C:\Users\DELL\AppData\Local\Temp\ipykernel_14012\2526601099.py:3: FutureWarning: The behavior of `series[i:j]` with an integer-dtype index is deprecated. In a future version, this will be treated as *label-based* indexing, consistent with e.g. `series[i]` lookups. To retain the old behavior, use `series.iloc[i:j]`. To get the future behavior, use `series.loc[i:j]`.
  over_sampling = round((y.value_counts()[0]/y.value_counts()[1:].sum())*y.value_counts()[1:]).to_dict()


{3: 527515, 4: 133634, 1: 98256, 5: 77378, 2: 63972, 6: 591}

In [9]:
from imblearn.over_sampling import RandomOverSampler
over = RandomOverSampler(sampling_strategy=over_sampling, random_state=42)
X_oversampled, y_oversampled = over.fit_resample(X, list(y))

In [10]:
y_oversampled = pd.Series(y_oversampled)

In [11]:
Counter(y_oversampled) # counts after oversampling

Counter({0: 901347,
         4: 133634,
         3: 527515,
         1: 98256,
         5: 77378,
         2: 63972,
         6: 591})

In [12]:
y_oversampled.value_counts()[1:].sum() # sum of counts of all benign attack records

C:\Users\DELL\AppData\Local\Temp\ipykernel_14012\3414190646.py:1: FutureWarning: The behavior of `series[i:j]` with an integer-dtype index is deprecated. In a future version, this will be treated as *label-based* indexing, consistent with e.g. `series[i]` lookups. To retain the old behavior, use `series.iloc[i:j]`. To get the future behavior, use `series.loc[i:j]`.
  y_oversampled.value_counts()[1:].sum() # sum of counts of all benign attack records


901346

In [13]:
# train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_oversampled, y_oversampled, test_size=0.3, stratify=y_oversampled)

In [14]:
y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

In [15]:
# min max scaling
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
# Principle Component Analysis
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95, random_state=0)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [17]:
variance_ratios = pca.explained_variance_ratio_

In [18]:
variance_ratios

array([0.47560811, 0.15581774, 0.1112682 , 0.05759421, 0.05250821,
       0.0434831 , 0.03037912, 0.02521105])

In [19]:
# converting to dataframe
X_train_pca = pd.DataFrame(X_train_pca)
X_test_pca=pd.DataFrame(X_test_pca)

In [20]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()

In [21]:
y_train=y_train.astype('int')
y_test=y_test.astype('int')

In [22]:
import timeit
training_time = timeit.timeit(lambda: model.fit(X_train_pca, y_train), number=1) 

In [23]:
start_time = timeit.default_timer()
y_pred = model.predict(X_test_pca)
testing_time = timeit.default_timer() - start_time

In [24]:
print("Average Training Time:", training_time) 
print("Testing Time:", testing_time)

Average Training Time: 571.3865601
Testing Time: 12.179635699999949


In [25]:
# Confusion Matrix
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, fbeta_score
confusion_matrix(y_test, y_pred)

array([[266952,     19,      0,     31,     17,   3382,      3],
       [     0,  29477,      0,      0,      0,      0,      0],
       [     0,      0,  19192,      0,      0,      0,      0],
       [     2,      0,      0, 158253,      0,      0,      0],
       [     3,      0,      0,      0,  40087,      0,      0],
       [  2055,      8,      0,      0,      0,  21150,      0],
       [    26,      0,      0,      0,      0,      0,    151]],
      dtype=int64)

In [26]:
print("Accuracy: ")
accuracy_score(y_test, y_pred)

Accuracy: 


0.9897449741867724

In [27]:
print("Precision: ")
precision_score(y_test, y_pred, average='weighted')

Precision: 


0.990060863102983

In [28]:
print("Recall: ")
recall_score(y_test, y_pred, average='weighted')

Recall: 


0.9897449741867724

In [29]:
print("F2 Score: ")
fbeta_score(y_test, y_pred, beta=2, average='weighted')

F2 Score: 


0.9897863218450031

In [30]:
# CASE 2
# finding required counts of all attack record types
import math
count = math.ceil(y.value_counts()[0]/6)
count

150225

In [31]:
count_of_benign = count*6

In [32]:
over_sampling = {0: count_of_benign}
for i in range(1, 7): over_sampling[i] = count

In [33]:
over = RandomOverSampler(sampling_strategy=over_sampling, random_state=42)

X_resampled, y_resampled = over.fit_resample(X, list(y))

In [34]:
y_resampled = pd.Series(y_resampled)

In [35]:
y_resampled.value_counts()

0    901350
4    150225
3    150225
1    150225
5    150225
2    150225
6    150225
dtype: int64

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, stratify=y_resampled)

In [37]:
y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

In [38]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [39]:
pca = PCA(n_components=0.95, random_state=0)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [40]:
variance_ratios = pca.explained_variance_ratio_

In [41]:
variance_ratios

array([0.41742953, 0.1759884 , 0.11876605, 0.06437424, 0.05200011,
       0.04978848, 0.03173179, 0.02265044, 0.01888183])

In [42]:
X_train_pca = pd.DataFrame(X_train_pca)
X_test_pca=pd.DataFrame(X_test_pca)

In [43]:
model = RandomForestClassifier()

In [44]:
y_train=y_train.astype('int')
y_test=y_test.astype('int')

In [45]:
training_time = timeit.timeit(lambda: model.fit(X_train_pca, y_train), number=1) 

In [46]:
start_time = timeit.default_timer()
y_pred = model.predict(X_test_pca)
testing_time = timeit.default_timer() - start_time

In [47]:
print("Average Training Time:", training_time) 
print("Testing Time:", testing_time)

Average Training Time: 1047.9917252
Testing Time: 15.886067599999933


In [48]:
confusion_matrix(y_test, y_pred)

array([[266053,     21,      2,     16,     15,   4178,    120],
       [     0,  45061,      0,      0,      0,      7,      0],
       [     0,      0,  45067,      0,      0,      0,      0],
       [     7,      0,      0,  45060,      0,      0,      0],
       [     0,      0,      0,      0,  45068,      0,      0],
       [  1608,      7,      0,      0,      0,  43452,      0],
       [     0,      0,      0,      0,      0,      0,  45068]],
      dtype=int64)

In [49]:
print("Accuracy: ")
accuracy_score(y_test, y_pred)

Accuracy: 


0.9889406630794549

In [50]:
print("Precision: ")
precision_score(y_test, y_pred, average='weighted')

Precision: 


0.9893282398042026

In [51]:
print("Recall: ")
recall_score(y_test, y_pred, average='weighted')

Recall: 


0.9889406630794549

In [52]:
print("F2 Score: ")
fbeta_score(y_test, y_pred, beta=2, average='weighted')

F2 Score: 


0.9889708118741044